In [2]:

# ! pip install -r ../../../../requirements.txt -U

In [3]:
from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from azure.identity import AzureCliCredential

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.core.exceptions import ResourceNotFoundError



In [4]:

import os
import base64
from dotenv import load_dotenv

In [5]:
AgentName = "Search-Agent"
AgentInstructions = """You are a helpful assistant that can search the web for current information.
            Use the Bing search tool to find up-to-date information and provide accurate, well-sourced answers.
            Always cite your sources when possible. Add something fun at the end of your answer."""

In [6]:
# 1. Resolve the Foundry project connection ID from connection name candidates
env_connection_name = os.environ["BING_CONNECTION_NAME"]

with AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
) as project:
    bing_connection = None
    try:
        bing_connection = project.connections.get(env_connection_name)
        print(f"Resolved connection: {bing_connection.name} -> {bing_connection.id}")
    except ResourceNotFoundError:
        pass

    if bing_connection is None:
        available = [c.name for c in project.connections.list()]
        raise RuntimeError(
            f"No matching Bing connection found. Tried: {env_connection_name}. "
            f"Available project connections: {available}"
        )

    # 2. Check if agent exists and get latest version
    agent_name = AgentName
    latest_version = None
    
    try:
        # Try to get the latest version of the agent
        versions = list(project.agents.list_versions(agent_name=agent_name))
        if versions:
            latest_agent = versions[0]  # Get the latest version
            latest_version = int(latest_agent.version) if hasattr(latest_agent, 'version') else latest_agent.version
            next_version = latest_version + 1
            print(f"Agent '{agent_name}' exists. Latest version: {latest_version}. Creating new version: {next_version}")
    except ResourceNotFoundError:
        print(f"Agent '{agent_name}' does not exist. Creating new agent with version: {next_version}")

    # 3. Create a new agent version with Bing grounding tool attached (sync path)
    model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME")
    agent = project.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=AgentInstructions,
            tools=[
                BingGroundingTool(
                    bing_grounding=BingGroundingSearchToolParameters(
                        search_configurations=[
                            BingGroundingSearchConfiguration(
                                project_connection_id=bing_connection.id
                            )
                        ]
                    )
                )
            ],
        ),
        description="Bing grounded search agent",
    )

    print(f"\n✓ Agent version {agent.version} created successfully")
    print(f"  - Agent ID: {agent.id}")
    print(f"  - Agent Name: {agent.name}")
    print(f"  - Version: {agent.version}")
    if latest_version:
        print(f"  - Previous version: {latest_version}")

Resolved connection: hanabingsearchj9kdyn -> /subscriptions/d6233897-5c9f-47f9-8507-6d4ada2d5183/resourceGroups/AgenticAI-AzureMasterclass/providers/Microsoft.CognitiveServices/accounts/hana-foundry-test/projects/proj-default/connections/hanabingsearchj9kdyn
Agent 'Search-Agent' exists. Latest version: 2. Creating new version: 3

✓ Agent version 2 created successfully
  - Agent ID: Search-Agent:2
  - Agent Name: Search-Agent
  - Version: 2
  - Previous version: 2


In [7]:
UserQuestion = "when was the city of Casablanca founded"

In [8]:
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
)

openai_client = project_client.get_openai_client()

# Reference the agent to get a response
response = openai_client.responses.create(
    input=[{"role": "user", "content": UserQuestion}],
    extra_body={"agent_reference": {"name": AgentName, "version": agent.version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")

Response output: Casablanca doesn’t have a single clear “founding year” the way some younger cities do.

- The **site was originally a Berber settlement** called **Anfa**, which sources place around the **12th century** (1100s CE).【5:2†source】【5:4†source】  
- Earlier still, the wider area was settled by Berbers by about the **10th century BC**, used later by Phoenicians and Romans as a port, but that’s more about the region than the continuous city of today.【5:0†source】  
- The **modern town that became “Casablanca”** was substantially **rebuilt in the late 18th century** by Sultan Sīdī Muhammad ibn ʿAbd Allāh after the 1755 earthquake; European traders then popularized the Spanish name “Casablanca.”【5:4†source】

So, historically:
- Origin as Anfa (village/city): **12th century**
- Rebuilt and set on the path to becoming today’s Casablanca: **late 1700s**

Fun bit: long before it was famous for a black-and-white movie, the same coastline was notorious as a **pirate base harassing Chris